# Step 09.5 — Why LSTM/GRU Still Did Not Solve Everything：门控之后还剩下什么问题？

Step 9.3–9.4 已经看到，LSTM 和 GRU 都通过 gates 为旧状态提供了可学习的保留路径，改善 Vanilla RNN 的长期 credit assignment。

但 **better gradient flow 不等于 recurrence 的所有结构性问题都消失**。

这一节只收束 recurrent family 仍然存在的三类核心限制：

1. time-step serial dependency（时间步串行依赖）仍然存在；
2. long information path / fixed-width state bottleneck（长信息路径与固定宽度状态压缩）仍然存在；
3. GPU 可以并行一个时间步内部的矩阵运算，但不能把有真实依赖的所有 T 个 recurrent updates 同时完成。

本轮只准备 Step 9.5；不进入 Step 10.1。


## 1. 先说清楚：LSTM / GRU 到底解决了什么，又没有改变什么？

Vanilla RNN 的主要困难之一，是长期 gradient contribution 反复穿过 recurrent transform 与 activation derivative。

LSTM 引入：

```text
cell state c_t
forget / input / output gates
```

GRU 引入：

```text
update gate z_t
reset gate r_t
```

它们共同带来的关键改进是：**模型可以学习什么时候更直接地保留旧状态，什么时候写入新信息。**

例如 LSTM direct cell path：

$$
c_t=f_t\odot c_{t-1}+i_t\odot g_t
$$

以及 GRU direct retain path：

$$
h_t=(1-z_t)\odot n_t+z_t\odot h_{t-1}.
$$

当某些 coordinates 的 `f_t` 或 `z_t` 接近 1 时，对应 direct path 可以接近 identity，因此长期 information / gradient contribution 比 Vanilla RNN 更容易保留。

但是门控并没有改变一个最基本的事实：

```text
当前 recurrent state 仍然依赖前一时刻 recurrent state。
```

LSTM：

$$
(h_t,c_t)=F(x_t,h_{t-1},c_{t-1})
$$

GRU：

$$
h_t=G(x_t,h_{t-1}).
$$

所以 gates 改善了**状态怎样更新**，却没有消除**时间依赖图本身**。


## 2. Time-Step Serial Dependency：T 个时间步仍然形成 Critical Path

先解释一个术语：**critical path（关键依赖路径）**，指完成最终结果之前必须按依赖关系依次完成的最长计算链。

对于长度为 T 的 recurrent sequence：

```text
h_-1
 ↓
h_0
 ↓
h_1
 ↓
...
 ↓
h_(T-1)
```

要算 `h_5`，必须先有 `h_4`；要有 `h_4`，又必须先有 `h_3`。

因此沿 Time axis 的依赖深度大约随：

$$
O(T)
$$

增长。

这里的 `O(T)` 只表示：当其他条件固定时，必须经过的 recurrent dependency steps 与 T 同阶增长；它不是 wall-clock 秒数。

### Batch 能并行，但 Time 仍然有依赖

固定某个 t 时，B 个 samples 可以一起做矩阵运算：

```text
x_t      : (B,C)
h_prev   : (B,H)
```

所以 Batch axis 仍然能很好地利用并行硬件。

但同一条 sequence 的：

```text
t=0,1,2,...,T-1
```

不能简单当成 T 个彼此独立的 recurrent updates。

这和你在 Step 8.3 学过的 Vanilla RNN 完全一致：**LSTM / GRU 改了 cell 内部公式，没有删除跨时间的 recurrent edge。**


In [ ]:
import matplotlib.pyplot as plt

T = 8
time_steps = list(range(T))
dependency_depth = [t + 1 for t in time_steps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(time_steps, dependency_depth, marker='o')
ax.set_xlabel('time index t')
ax.set_ylabel('recurrent dependency depth')
ax.set_title('Recurrent critical path grows with sequence length')
ax.grid(True, alpha=0.25)
plt.show()


## 3. 不是“RNN 完全不能并行”：能提前算什么，不能提前算什么？

这里要避免另一个常见过度简化：

> 不能说 LSTM / GRU 的所有计算都必须完全串行。

以 LSTM 为例，输入 `X=(B,T,C)` 对 packed input weight `Wx=(4H,C)` 的 projection：

$$
XW_x^T
$$

可以一次得到：

$$
(B,T,4H).
$$

GRU 类似，可以一次得到 `(B,T,3H)` 的 input-side projection。

这些部分只依赖 `X`，不需要等待上一时刻 hidden state，因此可以跨 T 批量计算。

但是 gate 的 recurrent contribution 仍然需要：

```text
h_(t-1)
```

而这个值只有前一个时间步真正完成后才知道。

因此更准确地说：

```text
input-side projection      -> 可以跨多个 t 并行/批量计算
recurrent state transition -> 沿 T 仍有 serial dependency
```

下面的静态 MLX 示例只展示 shape，不做性能 benchmark。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

B, T, C, H = 2, 6, 3, 4
X = mx.zeros((B, T, C))

lstm = nn.LSTM(input_size=C, hidden_size=H, bias=True)
gru = nn.GRU(input_size=C, hidden_size=H, bias=True)

# These input-side projections depend only on X.
lstm_input_projection = X @ lstm.Wx.T   # (B,T,4H)
gru_input_projection = X @ gru.Wx.T     # (B,T,3H)

print('LSTM input projection:', lstm_input_projection.shape)
print('GRU input projection :', gru_input_projection.shape)

# But recurrent gate/state updates still need h_(t-1) (and c_(t-1) for LSTM).


## 4. Gating 没有消除 Fixed-Width State Bottleneck，也没有让信息路径变短

LSTM / GRU 虽然更擅长决定“保留什么、忘掉什么”，但它们仍然把已经读过的历史压缩进固定宽度 recurrent states。

GRU：

$$
h_t\in\mathbb R^H
$$

LSTM：

$$
h_t,c_t\in\mathbb R^H.
$$

无论已经处理 10 个 token 还是 1000 个 token，下一时刻向前传递的 recurrent state width 仍然由 H 决定。

这在工程上很有价值：streaming 时不需要把完整过去全部重新读一遍。

但它也形成一个实际的 **fixed-width state bottleneck（固定宽度状态瓶颈）**：模型必须持续决定哪些历史信息值得保存在有限表示里。

### 这里不能做过强的数学断言

不能只因为：

```text
历史很长
state width 固定为 H
```

就宣称“数学上绝对不可能编码全部历史”。理想实数可以承载复杂编码。

真实神经网络的问题来自更现实的约束组合：

```text
有限精度
有限 H
有限参数
连续可训练的更新函数
噪声与优化困难
需要从 state 中可用地恢复任务相关信息
```

所以这里说的是**实际 representation bottleneck**，不是纯维度计数证明。

此外，即便 gate 帮助某些信息在 state 中保留，一个很早 token 对很晚 output 的影响仍然要经历很多 recurrent transitions。Gating 改善每一步的传递条件，却没有把路径长度从 O(T) 变成 O(1)。


In [ ]:
B, H = 2, 4

for T_example in [4, 40, 400]:
    gru_state_shape = (B, H)
    lstm_hidden_shape = (B, H)
    lstm_cell_shape = (B, H)

    print(
        f'T={T_example:3d} | '
        f'GRU state={gru_state_shape} | '
        f'LSTM h={lstm_hidden_shape}, c={lstm_cell_shape}'
    )


## 5. 为什么这些限制在 GPU 时代尤其重要？并收束整个 Step 9

GPU 很擅长大量规则、彼此独立的矩阵运算。

在 LSTM / GRU 的一个固定时间步内部，可以对：

```text
Batch rows
hidden coordinates
packed gate projections
```

做大量并行计算。

问题在于 Time axis 上仍然有：

```text
state_(t-1)
   ↓
state_t
   ↓
state_(t+1)
```

这种真实依赖。

因此长 sequence 会带来两个不同维度的成本：

1. **总计算量增加**：T 越大，需要做更多 recurrent updates；
2. **sequential critical path 变长**：后一个 recurrent step 必须等待前一个 state。

这意味着即使单个时间步内部的矩阵运算很快，长序列训练仍然难以像“很多彼此独立的位置计算”那样充分展开到整个 Time axis。

### Step 9 到这里形成的完整历史链

```text
Vanilla RNN
↓
显式 recurrent memory
↓
BPTT
↓
long temporal Jacobian products
↓
vanishing / exploding gradient risk
↓
LSTM / GRU gates
↓
更可控的 information / gradient retention
↓
但 recurrence 仍存在
↓
Time axis 仍串行
fixed-width state 仍需压缩历史
long information paths 仍存在
```

因此 LSTM / GRU 解决的是 recurrent learning 的重要一部分，而不是把 sequence modeling 的全部问题一次解决。

roadmap 的下一阶段会从这些剩余限制继续发展，但本节到此停止，不进入 Step 10.1。


### Step 9.5 材料边界

- LSTM / GRU 的 gates 改善长期 information / gradient flow，但没有消除 recurrence。
- 同一 sequence 的 recurrent states 仍形成长度随 T 增长的 serial dependency chain。
- Batch axis 与单步内部矩阵运算可以并行；不能把“时间递推串行”误写成“RNN 所有运算都不能并行”。
- input-side packed projections 可以跨 T 批量计算，但 hidden-dependent recurrent update 必须等待 previous state。
- LSTM / GRU 仍把历史压缩进固定宽度 recurrent states；这是实际 representation bottleneck，但不是简单维度计数即可证明的绝对信息不可能性。
- Gating 可以改善局部传递条件，但没有把远距离 information path 缩短成常数长度。
- 本轮只准备 Step 9.5；不进入 Step 10.1，也不据此声明学员已经掌握或完成练习。
